# EDA d'inspection — french-second-hand-cars

**Dataset candidat #1** pour le pilier Prix. Cible = `price`.

> ⚠️ **Gate EDA** (cf. `ml/AGENTS.md`) : ce notebook **inspecte** seulement.
> Aucun nettoyage, aucune modélisation. Le fichier `raw/` n'est jamais modifié.
> À la fin : résumé chiffré → **STOP** → validation avant toute suite.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../../data/french-second-hand-cars/raw/French_second_hand_cars.csv")
df = pd.read_csv(RAW)
print(f"{df.shape[0]} lignes x {df.shape[1]} colonnes")

2441 lignes x 40 colonnes


## 1. Colonnes & types

In [3]:
print("Colonnes:")
for c in df.columns:
    print(f"  - {c}")
print()
df.dtypes

Colonnes:
  - publishedsince
  - carmodel
  - price
  - année
  - miseencirculation
  - contrôletechnique
  - kilométragecompteur
  - énergie
  - boîtedevitesse
  - couleurextérieure
  - nombredeportes
  - nombredeplaces
  - garantie
  - premièremain(déclaratif)
  - nombredepropriétaires
  - puissancefiscale
  - puissancedin
  - crit'air
  - émissionsdeco2
  - consommationmixte
  - normeeuro
  - options
  - departement
  - id
  - waranty
  - vendeur
  - vérifié&garanti
  - rechargeable
  - autonomiebatterie
  - capacitébatterie
  - conso.batterie
  - couleurintérieure
  - puissancemoteur
  - primeàlaconversion
  - garantieconstructeur
  - provenance
  - prixinclutlabatterie
  - voltagebatterie
  - intensitébatterie
  - prixinclutlabatterie.1



publishedsince                  str
carmodel                        str
price                           str
année                       float64
miseencirculation               str
contrôletechnique               str
kilométragecompteur             str
énergie                         str
boîtedevitesse                  str
couleurextérieure               str
nombredeportes              float64
nombredeplaces              float64
garantie                        str
premièremain(déclaratif)        str
nombredepropriétaires       float64
puissancefiscale                str
puissancedin                    str
crit'air                    float64
émissionsdeco2                  str
consommationmixte               str
normeeuro                       str
options                         str
departement                   int64
id                            int64
waranty                         str
vendeur                         str
vérifié&garanti                 str
rechargeable                

In [4]:
df.head(3)

,publishedsince,carmodel,price,année,miseencirculation,contrôletechnique,kilométragecompteur,énergie,boîtedevitesse,couleurextérieure,...,conso.batterie,couleurintérieure,puissancemoteur,primeàlaconversion,garantieconstructeur,provenance,prixinclutlabatterie,voltagebatterie,intensitébatterie,prixinclutlabatterie.1
0,2 jours,\n RENAULT TWINGO 3\n,\n 11 080 €\n,2020.0,17/07/2020,non requis,27 297 Km,Essence,mécanique,gris,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5 jours,\n BMW SERIE 4 F36 GRAN COUPE\n,\n 50 690 €\n,2019.0,27/04/2019,non requis,59 778 Km,Diesel,automatique,Saphirschwarz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19 jours,\n BMW SERIE 2 F45 ACTIVE TOURER\n,\n 19 740 €\n,2017.0,14/05/2017,requis,128 835 Km,Hybride essence électrique,automatique,gris metal,...,11 kWh/100km,cuir noir,165 kW,,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Valeurs manquantes (% par colonne)

In [5]:
na_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
na_pct

prixinclutlabatterie.1      99.9
intensitébatterie           97.6
conso.batterie              97.3
autonomiebatterie           95.9
voltagebatterie             95.5
provenance                  95.2
prixinclutlabatterie        94.2
capacitébatterie            93.6
rechargeable                92.4
primeàlaconversion          83.2
puissancemoteur             82.9
vérifié&garanti             82.3
garantieconstructeur        71.5
nombredepropriétaires       60.3
couleurintérieure           53.1
garantie                    15.5
consommationmixte           13.0
émissionsdeco2               8.5
waranty                      7.7
nombredeplaces               3.2
crit'air                     3.2
normeeuro                    3.1
puissancedin                 2.9
puissancefiscale             0.3
nombredeportes               0.2
departement                  0.0
énergie                      0.0
price                        0.0
année                        0.0
miseencirculation            0.0
contrôlete

## 3. Qualité de la cible `price`

Aujourd'hui `price` est du **texte sale** (retours ligne, `€`, espaces insécables `\xa0`).
On teste combien de valeurs seraient convertibles en nombre — **sans écrire** le résultat.

In [6]:
price_raw = df["price"].astype(str)
print("Exemple brut:", repr(price_raw.iloc[0]))

price_test = (price_raw
              .str.replace("\xa0", "", regex=False)
              .str.replace("€", "", regex=False)
              .str.replace(" ", "", regex=False)
              .str.strip())
price_num = pd.to_numeric(price_test, errors="coerce")

n = len(df)
ok = price_num.notna().sum()
print(f"Convertibles en nombre : {ok}/{n} ({ok/n*100:.1f}%)")
print(f"Non convertibles       : {n-ok}")
print()
print("Stats prix (si convertis) :")
price_num.describe().round(0)

Exemple brut: '\n          11 080\xa0€\n        '
Convertibles en nombre : 2392/2441 (98.0%)
Non convertibles       : 49

Stats prix (si convertis) :


count      2392.0
mean      33367.0
std       28699.0
min        3020.0
25%       18130.0
50%       25495.0
75%       37190.0
max      295070.0
Name: price, dtype: float64

## 4. Aberrations & doublons

In [7]:
print("Lignes dupliquées :", df.duplicated().sum())

dup_like = [c for c in df.columns if c.endswith(".1") or "prixinclutlabatterie" in c]
print("Colonnes suspectes (doublon source) :", dup_like)

print("Prix <= 0 :", int((price_num <= 0).sum()))
if "année" in df.columns:
    an = pd.to_numeric(df["année"], errors="coerce")
    print("Année hors [1980, 2026] :", int(((an < 1980) | (an > 2026)).sum()))

Lignes dupliquées : 0
Colonnes suspectes (doublon source) : ['prixinclutlabatterie', 'prixinclutlabatterie.1']
Prix <= 0 : 0
Année hors [1980, 2026] : 11


## 5. Cardinalité des catégorielles clés

In [8]:
for col in ["énergie", "boîtedevitesse", "carmodel", "departement"]:
    if col in df.columns:
        print(f"--- {col} : {df[col].nunique()} valeurs distinctes ---")
        print(df[col].value_counts(dropna=False).head(5))
        print()

--- énergie : 7 valeurs distinctes ---
énergie
Diesel                        1156
Essence                       1093
Hybride essence électrique     134
Electrique                      40
Hybride diesel électrique       12
Name: count, dtype: int64

--- boîtedevitesse : 3 valeurs distinctes ---
boîtedevitesse
 automatique    1278
 mécanique      1160
                   2
NaN                1
Name: count, dtype: int64

--- carmodel : 585 valeurs distinctes ---
carmodel
\n    PEUGEOT 3008 (2E GENERATION)\n      75
\n    CITROEN C3 (3E GENERATION)\n        58
\n    RENAULT CLIO 5\n                    36
\n    RENAULT CAPTUR 2\n                  33
\n    PEUGEOT 308 (2E GENERATION) SW\n    32
Name: count, dtype: int64

--- departement : 63 valeurs distinctes ---
departement
24    217
69    169
31    124
78    121
33    107
Name: count, dtype: int64



## 6. Résumé chiffré (→ STOP validation)

In [9]:
na_global = df.isna().mean().mean() * 100
tres_vides = (na_pct > 90).sum()

print("=" * 50)
print("RÉSUMÉ — french-second-hand-cars")
print("=" * 50)
print(f"Dimensions        : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"NA global moyen    : {na_global:.1f} %")
print(f"Colonnes >90% vide : {tres_vides}")
print(f"Cible price OK     : {ok}/{n} convertibles ({ok/n*100:.1f}%)")
print(f"Lignes dupliquées  : {df.duplicated().sum()}")
print(f"Colonne dupliquée  : {dup_like}")
print("=" * 50)
print("STOP — attente validation avant nettoyage / régression.")

RÉSUMÉ — french-second-hand-cars
Dimensions        : 2441 lignes x 40 colonnes
NA global moyen    : 33.8 %
Colonnes >90% vide : 9
Cible price OK     : 2392/2441 convertibles (98.0%)
Lignes dupliquées  : 0
Colonne dupliquée  : ['prixinclutlabatterie', 'prixinclutlabatterie.1']
STOP — attente validation avant nettoyage / régression.
